# 🫀 CardioIA — Visão Computacional
## Classificação de Imagens de ECG com Redes Neurais Convolucionais

**FIAP — Faculdade de Informática e Administração Paulista**

---

### 📋 Sumário
1. [Configuração do Ambiente](#config)
2. [Carregamento e Exploração do Dataset](#dataset)
3. [Pré-processamento de Imagens](#preprocessing)
4. [Divisão Treino / Validação / Teste](#split)
5. [Parte 1 — Relatório de Pré-processamento](#report1)
6. [CNN Simples (Treinada do Zero)](#cnn)
7. [Transfer Learning com VGG16](#vgg16)
8. [Avaliação e Comparação de Modelos](#evaluation)
9. [Protótipo Interativo de Classificação](#prototype)
10. [Conclusão](#conclusion)

---
## 1. ⚙️ Configuração do Ambiente <a id="config"></a>

Instalação das dependências necessárias e importação das bibliotecas.

In [ ]:
# Instalação das dependências (Google Colab)
!pip install -q tensorflow matplotlib seaborn scikit-learn pillow opencv-python-headless

In [ ]:
import os
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

warnings.filterwarnings('ignore')

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

---
## 2. 📂 Carregamento e Exploração do Dataset <a id="dataset"></a>

O dataset utilizado é o [ECG Image Dataset](https://www.kaggle.com/datasets/evilspirit05/ecg-analysis), composto por imagens de eletrocardiogramas (ECG) categorizadas em 4 classes clínicas:

| Classe | Descrição | Prefixo |
|---|---|---|
| **Normal** | ECG com ritmo sinusal normal | `Normal` |
| **Infarto do Miocárdio** | Infarto agudo do miocárdio | `MI` |
| **Batimento Anormal** | Arritmias e batimentos anormais | `HB` |
| **Histórico de Infarto** | Histórico de infarto prévio | `PMI` |

Este repositório contém uma **amostra representativa** de 30 imagens por classe (120 total). Para treinamento mais robusto, recomendamos utilizar o dataset completo do Kaggle.

In [ ]:
# ============================================================
# CONFIGURAÇÃO DE CAMINHOS
# ============================================================
# Ajuste o BASE_PATH conforme seu ambiente:
# - Google Colab: monte o Drive ou clone o repo
# - Local: use o caminho absoluto do repositório
# ============================================================

# Para Google Colab (descomente se necessário):
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_PATH = '/content/drive/MyDrive/CardioIA/datasets/images'

# Para execução local:
BASE_PATH = '../datasets/images'

CLASS_MAP = {
    'normal_ecg_images': 'Normal',
    'myocardial_infarction_ecg_images': 'Infarto do Miocárdio',
    'abnormal_heartbeat_ecg_images': 'Batimento Anormal',
    'post_mi_history_ecg_images': 'Histórico de Infarto'
}

CLASS_DIRS = list(CLASS_MAP.keys())
CLASS_NAMES = list(CLASS_MAP.values())
NUM_CLASSES = len(CLASS_NAMES)

print(f"Número de classes: {NUM_CLASSES}")
print(f"Classes: {CLASS_NAMES}")
print(f"Base path: {os.path.abspath(BASE_PATH)}")

In [ ]:
# ============================================================
# EXPLORAÇÃO DO DATASET
# ============================================================
image_paths = []
image_labels = []

for idx, class_dir in enumerate(CLASS_DIRS):
    class_path = os.path.join(BASE_PATH, class_dir)
    if not os.path.exists(class_path):
        print(f"⚠️ Diretório não encontrado: {class_path}")
        continue
    files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"📁 {CLASS_NAMES[idx]:25s} | {len(files):3d} imagens | Dir: {class_dir}")
    for f in sorted(files):
        image_paths.append(os.path.join(class_path, f))
        image_labels.append(idx)

print(f"\n📊 Total de imagens: {len(image_paths)}")

In [ ]:
# ============================================================
# DISTRIBUIÇÃO DAS CLASSES
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = Counter(image_labels)
colors = ['#2ecc71', '#e74c3c', '#f39c12', '#3498db']
axes[0].bar(CLASS_NAMES, [counts[i] for i in range(NUM_CLASSES)],
            color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribuição das Classes', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Quantidade de Imagens')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate([counts[i] for i in range(NUM_CLASSES)]):
    axes[0].text(i, v + 0.3, str(v), ha='center', fontweight='bold', fontsize=12)

axes[1].pie([counts[i] for i in range(NUM_CLASSES)], labels=CLASS_NAMES,
            autopct='%1.1f%%', colors=colors, startangle=90,
            textprops={'fontsize': 11})
axes[1].set_title('Proporção por Classe', fontsize=14, fontweight='bold')

plt.suptitle('📊 Análise do Dataset de ECG', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# AMOSTRAS DE CADA CLASSE (IMAGENS ORIGINAIS)
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('🖼️ Amostras de ECG por Classe (Antes do Pré-processamento)',
             fontsize=16, fontweight='bold')

for idx, class_dir in enumerate(CLASS_DIRS):
    class_path = os.path.join(BASE_PATH, class_dir)
    files = sorted([f for f in os.listdir(class_path)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

    for row in range(2):
        img = Image.open(os.path.join(class_path, files[row]))
        axes[row, idx].imshow(img)
        axes[row, idx].set_title(f"{CLASS_NAMES[idx]}\n({files[row]})", fontsize=10)
        axes[row, idx].axis('off')
        if row == 0:
            axes[row, idx].text(
                0.5, -0.15, f"Original: {img.size[0]}x{img.size[1]} px",
                transform=axes[row, idx].transAxes, ha='center',
                fontsize=9, color='gray'
            )

plt.tight_layout()
plt.show()

---
## 3. 🔧 Pré-processamento de Imagens <a id="preprocessing"></a>

O pipeline de pré-processamento segue as seguintes etapas:

1. **Redimensionamento**: Todas as imagens são redimensionadas para **224×224 pixels** (padrão para CNNs e compatível com VGG16)
2. **Normalização**: Os valores dos pixels são normalizados para o intervalo **[0, 1]** (divisão por 255)
3. **Conversão de formato**: Todas as imagens são convertidas para **RGB** (3 canais)
4. **Data Augmentation**: Aplicação de transformações para aumentar a diversidade do dataset de treino

> **Justificativa**: O redimensionamento para 224×224 garante a compatibilidade com arquiteturas pré-treinadas. A normalização melhora a convergência do treinamento. O data augmentation é crucial dado o tamanho limitado do dataset (120 imagens).

In [ ]:
# ============================================================
# PIPELINE DE PRÉ-PROCESSAMENTO
# ============================================================
IMG_SIZE = (224, 224)

def preprocess_image(img_path, target_size=IMG_SIZE):
    """
    Pipeline de pré-processamento para uma única imagem de ECG.

    Etapas:
    1. Carregamento da imagem
    2. Conversão para RGB (caso esteja em outro formato)
    3. Redimensionamento para target_size
    4. Normalização dos pixels para [0, 1]

    Returns:
        numpy array normalizado com shape (target_size[0], target_size[1], 3)
    """
    img = Image.open(img_path)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.LANCZOS)
    img_array = np.array(img, dtype=np.float32)
    img_array = img_array / 255.0
    return img_array

# Processar todas as imagens
print("🔄 Processando imagens...")
X_all = []
y_all = []

for i, (path, label) in enumerate(zip(image_paths, image_labels)):
    img_processed = preprocess_image(path)
    X_all.append(img_processed)
    y_all.append(label)
    if (i + 1) % 30 == 0:
        print(f"  Processadas: {i + 1}/{len(image_paths)} imagens")

X_all = np.array(X_all)
y_all = np.array(y_all)

print(f"\n✅ Pré-processamento concluído!")
print(f"   Shape dos dados: {X_all.shape}")
print(f"   Dtype: {X_all.dtype}")
print(f"   Valores - Min: {X_all.min():.4f}, Max: {X_all.max():.4f}, Média: {X_all.mean():.4f}")

In [ ]:
# ============================================================
# VISUALIZAÇÃO: ANTES vs DEPOIS DO PRÉ-PROCESSAMENTO
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('🔧 Comparação: Original vs Pré-processado',
             fontsize=16, fontweight='bold')

sample_indices = [0, 30, 60, 90]  # Uma imagem de cada classe

for col, idx in enumerate(sample_indices):
    # Original
    img_orig = Image.open(image_paths[idx])
    axes[0, col].imshow(img_orig)
    axes[0, col].set_title(
        f"Original\n{CLASS_NAMES[y_all[idx]]}\n{img_orig.size[0]}x{img_orig.size[1]} px",
        fontsize=10
    )
    axes[0, col].axis('off')

    # Pré-processado
    axes[1, col].imshow(X_all[idx])
    axes[1, col].set_title("Pré-processado\n224x224 px | Normalizado [0,1]", fontsize=10)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('ORIGINAL', fontsize=12, fontweight='bold',
                       rotation=0, labelpad=80)
axes[1, 0].set_ylabel('PROCESSADO', fontsize=12, fontweight='bold',
                       rotation=0, labelpad=80)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# DISTRIBUIÇÃO DOS PIXELS POR CLASSE
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('📈 Distribuição de Intensidade de Pixels por Classe',
             fontsize=14, fontweight='bold')

for idx in range(NUM_CLASSES):
    class_mask = y_all == idx
    class_pixels = X_all[class_mask].flatten()
    axes[idx].hist(class_pixels, bins=50, color=colors[idx],
                   alpha=0.7, edgecolor='white')
    axes[idx].set_title(CLASS_NAMES[idx], fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Intensidade do Pixel')
    axes[idx].set_ylabel('Frequência')
    axes[idx].set_xlim(0, 1)

plt.tight_layout()
plt.show()

---
## 4. ✂️ Divisão Treino / Validação / Teste <a id="split"></a>

O dataset é dividido seguindo a proporção:
- **Treino**: 70% (84 imagens)
- **Validação**: 15% (18 imagens)
- **Teste**: 15% (18 imagens)

Utilizamos `stratify` para manter a proporção das classes em cada subconjunto.

In [ ]:
# ============================================================
# DIVISÃO ESTRATIFICADA DO DATASET
# ============================================================

# Primeiro split: 70% treino, 30% temporário
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.30, random_state=SEED, stratify=y_all
)

# Segundo split: divide o temporário em 50/50 (val e test)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("📊 Divisão do Dataset:")
print(f"   Treino    : {X_train.shape[0]:3d} imagens ({X_train.shape[0]/len(X_all)*100:.1f}%) | Shape: {X_train.shape}")
print(f"   Validação : {X_val.shape[0]:3d} imagens ({X_val.shape[0]/len(X_all)*100:.1f}%) | Shape: {X_val.shape}")
print(f"   Teste     : {X_test.shape[0]:3d} imagens ({X_test.shape[0]/len(X_all)*100:.1f}%) | Shape: {X_test.shape}")

# Verificar distribuição das classes em cada split
print("\n📊 Distribuição das classes por conjunto:")
for name, labels in [('Treino', y_train), ('Validação', y_val), ('Teste', y_test)]:
    dist = Counter(labels)
    print(f"   {name:10s}: {dict(sorted(dist.items()))}")

In [ ]:
# ============================================================
# VISUALIZAÇÃO DA DISTRIBUIÇÃO POR SPLIT
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('📊 Distribuição das Classes por Conjunto',
             fontsize=14, fontweight='bold')

for ax, (name, labels) in zip(axes, [('Treino', y_train),
                                      ('Validação', y_val),
                                      ('Teste', y_test)]):
    dist = Counter(labels)
    bars = ax.bar(CLASS_NAMES,
                  [dist.get(i, 0) for i in range(NUM_CLASSES)],
                  color=colors, edgecolor='white')
    ax.set_title(f'{name} ({len(labels)} imagens)',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Quantidade')
    ax.tick_params(axis='x', rotation=20)
    for bar, count in zip(bars, [dist.get(i, 0) for i in range(NUM_CLASSES)]):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
                str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# DATA AUGMENTATION (PARA TREINO)
# ============================================================
# Crucial para datasets pequenos - aumenta a diversidade
# das amostras sem coletar mais dados
# ============================================================

train_datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False,  # ECGs não devem ser espelhados horizontalmente
    brightness_range=[0.9, 1.1],
    fill_mode='nearest'
)

# Visualizar augmentation
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('🔄 Exemplos de Data Augmentation',
             fontsize=14, fontweight='bold')

sample_img = X_train[0:1]
axes[0, 0].imshow(sample_img[0])
axes[0, 0].set_title('Original', fontsize=11, fontweight='bold')
axes[0, 0].axis('off')

for i in range(1, 5):
    aug_img = train_datagen.random_transform(sample_img[0])
    axes[0, i].imshow(np.clip(aug_img, 0, 1))
    axes[0, i].set_title(f'Augmentação {i}', fontsize=11)
    axes[0, i].axis('off')

# Segunda linha com outra imagem
idx2 = min(21, len(X_train) - 1)
sample_img2 = X_train[idx2:idx2+1]
axes[1, 0].imshow(sample_img2[0])
axes[1, 0].set_title('Original', fontsize=11, fontweight='bold')
axes[1, 0].axis('off')

for i in range(1, 5):
    aug_img = train_datagen.random_transform(sample_img2[0])
    axes[1, i].imshow(np.clip(aug_img, 0, 1))
    axes[1, i].set_title(f'Augmentação {i}', fontsize=11)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print("✅ Data Augmentation configurado com sucesso!")

---
## 5. 📄 Relatório de Pré-processamento <a id="report1"></a>

### Pipeline de Preparação dos Dados

#### 1. Dataset Selecionado
Foi selecionado o **ECG Image Dataset** disponível no [Kaggle](https://www.kaggle.com/datasets/evilspirit05/ecg-analysis), contendo imagens de eletrocardiogramas classificadas em 4 categorias clínicas. O repositório contém uma amostra de 120 imagens (30 por classe).

#### 2. Etapas do Pré-processamento

| Etapa | Técnica | Justificativa |
|---|---|---|
| **Conversão de formato** | RGB (3 canais) | Padronização para CNNs que esperam entrada RGB |
| **Redimensionamento** | 224×224 pixels (LANCZOS) | Tamanho padrão para VGG16 e compatível com CNNs; interpolação LANCZOS preserva qualidade |
| **Normalização** | Divisão por 255 → [0, 1] | Melhora convergência do gradiente durante o treinamento |
| **Data Augmentation** | Rotação (±10°), shift (10%), zoom (10%), brilho | Aumenta diversidade das amostras de treino sem dados adicionais |

#### 3. Divisão dos Dados
- **Treino (70%)**: Usado para ajustar os pesos da rede neural
- **Validação (15%)**: Monitora overfitting durante o treinamento
- **Teste (15%)**: Avaliação final e imparcial do modelo
- **Estratificação**: Mantém a proporção das classes em cada subconjunto

#### 4. Decisões de Design
- **Horizontal flip desabilitado**: ECGs possuem orientação temporal (esquerda → direita), inverter alteraria o significado clínico
- **Augmentation conservadora**: Transformações sutis para não distorcer padrões diagnósticos
- **LANCZOS**: Interpolação de alta qualidade para preservar detalhes nas ondas do ECG

---
## 6. 🧠 CNN Simples (Treinada do Zero) <a id="cnn"></a>

Primeira abordagem: construção de uma **Rede Neural Convolucional** do zero, com arquitetura customizada para a classificação de imagens de ECG.

**Arquitetura:**
- 3 blocos convolucionais (Conv2D + BatchNorm + MaxPool + Dropout)
- 1 camada densa com 256 neurônios
- Saída com softmax (4 classes)

In [ ]:
# ============================================================
# MODELO 1: CNN SIMPLES (DO ZERO)
# ============================================================

def build_simple_cnn(input_shape=(224, 224, 3), num_classes=4):
    """Constrói uma CNN simples para classificação de ECG."""
    model = models.Sequential([
        # Bloco 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                      input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Bloco 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Bloco 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Classificador
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Construir o modelo
cnn_model = build_simple_cnn(num_classes=NUM_CLASSES)

# Compilar
cnn_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Resumo da arquitetura
cnn_model.summary()

In [ ]:
# ============================================================
# TREINAMENTO DA CNN SIMPLES
# ============================================================
EPOCHS = 50
BATCH_SIZE = 16

# Callbacks
cnn_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-6, verbose=1
    )
]

# Treinar com data augmentation
train_generator = train_datagen.flow(
    X_train, y_train, batch_size=BATCH_SIZE, seed=SEED
)

print("🚀 Iniciando treinamento da CNN Simples...")
print(f"   Épocas: {EPOCHS} | Batch Size: {BATCH_SIZE}")
print(f"   Treino: {len(X_train)} | Validação: {len(X_val)}")
print("-" * 50)

cnn_history = cnn_model.fit(
    train_generator,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=cnn_callbacks,
    verbose=1
)

print("\n✅ Treinamento concluído!")

In [ ]:
# ============================================================
# CURVAS DE TREINAMENTO - CNN SIMPLES
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('📈 Curvas de Treinamento — CNN Simples',
             fontsize=14, fontweight='bold')

axes[0].plot(cnn_history.history['accuracy'],
             label='Treino', linewidth=2, color='#2ecc71')
axes[0].plot(cnn_history.history['val_accuracy'],
             label='Validação', linewidth=2, color='#e74c3c')
axes[0].set_title('Acurácia', fontsize=12)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Acurácia')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot(cnn_history.history['loss'],
             label='Treino', linewidth=2, color='#2ecc71')
axes[1].plot(cnn_history.history['val_loss'],
             label='Validação', linewidth=2, color='#e74c3c')
axes[1].set_title('Loss (Perda)', fontsize=12)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. 🏗️ Transfer Learning com VGG16 <a id="vgg16"></a>

Segunda abordagem: utilização do modelo **VGG16**, pré-treinado no ImageNet, com **Transfer Learning**.

**Estratégia:**
- Base convolucional do VGG16 com pesos congelados (feature extraction)
- Camadas densas customizadas para classificação de 4 classes de ECG
- Fine-tuning: descongelar as últimas camadas convolucionais após treinamento inicial

> **Vantagem**: Mesmo com poucas imagens, o modelo aproveita features visuais aprendidas em milhões de imagens do ImageNet.

In [ ]:
# ============================================================
# MODELO 2: TRANSFER LEARNING COM VGG16
# ============================================================

def build_vgg16_transfer(input_shape=(224, 224, 3), num_classes=4):
    """Constrói modelo com Transfer Learning usando VGG16."""
    # Base pré-treinada (congelada)
    base_model = VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # Congelar todas as camadas da base
    for layer in base_model.layers:
        layer.trainable = False

    # Adicionar classificador customizado
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])

    return model, base_model

# Construir
vgg_model, vgg_base = build_vgg16_transfer(num_classes=NUM_CLASSES)

vgg_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

vgg_model.summary()

# Contar parâmetros
trainable = sum([tf.reduce_prod(w.shape).numpy() for w in vgg_model.trainable_weights])
non_trainable = sum([tf.reduce_prod(w.shape).numpy() for w in vgg_model.non_trainable_weights])
print(f"\n📊 Parâmetros treináveis: {trainable:,}")
print(f"📊 Parâmetros não-treináveis: {non_trainable:,}")

In [ ]:
# ============================================================
# FASE 1: TREINAMENTO COM BASE CONGELADA
# ============================================================
print("🚀 Fase 1: Treinamento com base VGG16 congelada...")
print("-" * 50)

vgg_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-6, verbose=1
    )
]

vgg_history_1 = vgg_model.fit(
    train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=SEED),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=vgg_callbacks,
    verbose=1
)

print("\n✅ Fase 1 concluída!")

In [ ]:
# ============================================================
# FASE 2: FINE-TUNING (DESCONGELAR ÚLTIMAS CAMADAS)
# ============================================================
print("🔓 Fase 2: Fine-tuning — descongelando últimas 4 camadas da VGG16...")

# Descongelar últimas 4 camadas convolucionais
for layer in vgg_base.layers[-4:]:
    layer.trainable = True

# Recompilar com learning rate menor
vgg_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

trainable_layers = sum(1 for l in vgg_base.layers if l.trainable)
print(f"Camadas treináveis na base: {trainable_layers}")
print("-" * 50)

vgg_history_2 = vgg_model.fit(
    train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=SEED),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=20,
    validation_data=(X_val, y_val),
    callbacks=vgg_callbacks,
    verbose=1
)

print("\n✅ Fine-tuning concluído!")

In [ ]:
# ============================================================
# CURVAS DE TREINAMENTO - VGG16
# ============================================================
# Combinar históricos das duas fases
vgg_acc = vgg_history_1.history['accuracy'] + vgg_history_2.history['accuracy']
vgg_val_acc = vgg_history_1.history['val_accuracy'] + vgg_history_2.history['val_accuracy']
vgg_loss = vgg_history_1.history['loss'] + vgg_history_2.history['loss']
vgg_val_loss = vgg_history_1.history['val_loss'] + vgg_history_2.history['val_loss']
phase1_epochs = len(vgg_history_1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('📈 Curvas de Treinamento — VGG16 Transfer Learning',
             fontsize=14, fontweight='bold')

axes[0].plot(vgg_acc, label='Treino', linewidth=2, color='#3498db')
axes[0].plot(vgg_val_acc, label='Validação', linewidth=2, color='#e74c3c')
axes[0].axvline(x=phase1_epochs, color='gray', linestyle='--',
                alpha=0.7, label='Início Fine-tuning')
axes[0].set_title('Acurácia', fontsize=12)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Acurácia')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(vgg_loss, label='Treino', linewidth=2, color='#3498db')
axes[1].plot(vgg_val_loss, label='Validação', linewidth=2, color='#e74c3c')
axes[1].axvline(x=phase1_epochs, color='gray', linestyle='--',
                alpha=0.7, label='Início Fine-tuning')
axes[1].set_title('Loss (Perda)', fontsize=12)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. 📊 Avaliação e Comparação de Modelos <a id="evaluation"></a>

Ambos os modelos são avaliados no **conjunto de teste** (dados nunca vistos durante o treinamento) utilizando as seguintes métricas:

- **Acurácia**: Proporção de classificações corretas
- **Precisão (Precision)**: Dos classificados como positivos, quantos realmente são?
- **Recall (Sensibilidade)**: Dos positivos reais, quantos foram identificados?
- **F1-Score**: Média harmônica entre precisão e recall
- **Matriz de Confusão**: Visualização detalhada dos acertos e erros por classe

In [ ]:
# ============================================================
# AVALIAÇÃO NO CONJUNTO DE TESTE
# ============================================================

def evaluate_model(model, X_test, y_test, model_name, class_names):
    """Avalia um modelo e retorna métricas detalhadas."""
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f"\n{'='*60}")
    print(f"📊 RESULTADOS — {model_name}")
    print(f"{'='*60}")
    print(f"  Acurácia  : {acc:.4f} ({acc*100:.2f}%)")
    print(f"  Precisão  : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"{'='*60}")

    print(f"\n📋 Relatório de Classificação:")
    print(classification_report(y_test, y_pred,
                                target_names=class_names, zero_division=0))

    return y_pred, y_pred_proba, {
        'accuracy': acc, 'precision': prec,
        'recall': rec, 'f1': f1
    }

# Avaliar CNN Simples
print("🧠 Avaliando CNN Simples...")
cnn_pred, cnn_proba, cnn_metrics = evaluate_model(
    cnn_model, X_test, y_test, "CNN Simples", CLASS_NAMES
)

# Avaliar VGG16
print("\n🏗️ Avaliando VGG16 Transfer Learning...")
vgg_pred, vgg_proba, vgg_metrics = evaluate_model(
    vgg_model, X_test, y_test, "VGG16 Transfer Learning", CLASS_NAMES
)

In [ ]:
# ============================================================
# MATRIZES DE CONFUSÃO
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('🔍 Matrizes de Confusão', fontsize=16, fontweight='bold')

for ax, (y_pred, title) in zip(axes, [(cnn_pred, 'CNN Simples'),
                                       (vgg_pred, 'VGG16 Transfer Learning')]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                cbar_kws={'label': 'Contagem'},
                annot_kws={'size': 14, 'fontweight': 'bold'})
    ax.set_title(f'{title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predição', fontsize=11)
    ax.set_ylabel('Real', fontsize=11)
    ax.tick_params(axis='x', rotation=20)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# COMPARAÇÃO DOS MODELOS
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))

metrics_names = ['Acurácia', 'Precisão', 'Recall', 'F1-Score']
cnn_vals = [cnn_metrics['accuracy'], cnn_metrics['precision'],
            cnn_metrics['recall'], cnn_metrics['f1']]
vgg_vals = [vgg_metrics['accuracy'], vgg_metrics['precision'],
            vgg_metrics['recall'], vgg_metrics['f1']]

x = np.arange(len(metrics_names))
width = 0.35

bars1 = ax.bar(x - width/2, cnn_vals, width, label='CNN Simples',
               color='#2ecc71', edgecolor='white', linewidth=1.5)
bars2 = ax.bar(x + width/2, vgg_vals, width, label='VGG16 Transfer Learning',
               color='#3498db', edgecolor='white', linewidth=1.5)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('🏆 Comparação de Desempenho dos Modelos',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

for bar in list(bars1) + list(bars2):
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

winner = "VGG16 Transfer Learning" if vgg_metrics['f1'] > cnn_metrics['f1'] else "CNN Simples"
print(f"\n🏆 Melhor modelo (por F1-Score): {winner}")

In [ ]:
# ============================================================
# VISUALIZAÇÃO DE PREDIÇÕES NO CONJUNTO DE TESTE
# ============================================================
best_model = vgg_model if vgg_metrics['f1'] > cnn_metrics['f1'] else cnn_model
best_name = "VGG16" if vgg_metrics['f1'] > cnn_metrics['f1'] else "CNN Simples"
best_pred = vgg_pred if vgg_metrics['f1'] > cnn_metrics['f1'] else cnn_pred
best_proba = vgg_proba if vgg_metrics['f1'] > cnn_metrics['f1'] else cnn_proba

num_samples = min(12, len(X_test))
rows = (num_samples + 3) // 4
fig, axes = plt.subplots(rows, 4, figsize=(20, 5 * rows))
fig.suptitle(f'🔍 Predições do Melhor Modelo ({best_name}) no Conjunto de Teste',
             fontsize=16, fontweight='bold')

if rows == 1:
    axes = axes.reshape(1, -1)

for i in range(num_samples):
    row, col = i // 4, i % 4
    ax = axes[row, col]

    ax.imshow(X_test[i])
    real = CLASS_NAMES[y_test[i]]
    pred = CLASS_NAMES[best_pred[i]]
    conf = best_proba[i][best_pred[i]] * 100

    correct = y_test[i] == best_pred[i]
    color = '#2ecc71' if correct else '#e74c3c'
    symbol = '✅' if correct else '❌'

    ax.set_title(f"{symbol} Real: {real}\nPred: {pred} ({conf:.1f}%)",
                fontsize=10, color='black',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.3))
    ax.axis('off')

# Esconder eixos extras
for i in range(num_samples, rows * 4):
    axes[i // 4, i % 4].axis('off')

plt.tight_layout()
plt.show()

---
## 9. 🖥️ Protótipo Interativo de Classificação <a id="prototype"></a>

Protótipo funcional que permite classificar novas imagens de ECG utilizando o melhor modelo treinado. O protótipo exibe a predição, o nível de confiança e um gráfico com as probabilidades para cada classe.

In [ ]:
# ============================================================
# PROTÓTIPO INTERATIVO DE CLASSIFICAÇÃO
# ============================================================

def classificar_ecg(img_path, modelo=None, nome_modelo=None):
    """
    Protótipo de classificação de ECG.
    Recebe o caminho de uma imagem e retorna a classificação
    com visualização detalhada.
    """
    if modelo is None:
        modelo = best_model
    if nome_modelo is None:
        nome_modelo = best_name

    # Pré-processar
    img_processed = preprocess_image(img_path)
    img_batch = np.expand_dims(img_processed, axis=0)

    # Predição
    proba = modelo.predict(img_batch, verbose=0)[0]
    pred_class = np.argmax(proba)
    confidence = proba[pred_class] * 100

    # Visualização
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle('🫀 CardioIA — Classificação de ECG',
                 fontsize=16, fontweight='bold')

    # Imagem original
    img_orig = Image.open(img_path)
    axes[0].imshow(img_orig)
    axes[0].set_title('Imagem Original', fontsize=12)
    axes[0].axis('off')

    # Imagem processada
    axes[1].imshow(img_processed)
    axes[1].set_title('Imagem Pré-processada (224x224)', fontsize=12)
    axes[1].axis('off')

    # Gráfico de probabilidades
    bar_colors = ['#2ecc71' if i == pred_class else '#bdc3c7'
                  for i in range(NUM_CLASSES)]
    bars = axes[2].barh(CLASS_NAMES, proba * 100, color=bar_colors,
                        edgecolor='white', linewidth=1.5)
    axes[2].set_xlabel('Confiança (%)', fontsize=11)
    axes[2].set_title('Probabilidades por Classe', fontsize=12)
    axes[2].set_xlim(0, 110)
    for bar, p in zip(bars, proba):
        axes[2].text(bar.get_width() + 1,
                    bar.get_y() + bar.get_height()/2,
                    f'{p*100:.1f}%', va='center',
                    fontweight='bold', fontsize=11)

    plt.tight_layout()
    plt.show()

    # Resultado textual
    print(f"\n{'='*50}")
    print(f"🔬 RESULTADO DA CLASSIFICAÇÃO")
    print(f"{'='*50}")
    print(f"  Modelo utilizado : {nome_modelo}")
    print(f"  Arquivo          : {os.path.basename(img_path)}")
    print(f"  Classificação    : {CLASS_NAMES[pred_class]}")
    print(f"  Confiança        : {confidence:.2f}%")
    print(f"{'='*50}")

    # Alerta clínico
    if pred_class != 0:  # Não é Normal
        print(f"\n  ⚠️  ATENÇÃO: Padrão anormal detectado!")
        print(f"      Recomenda-se avaliação médica especializada.")
    else:
        print(f"\n  ✅ ECG dentro dos padrões normais.")

    return CLASS_NAMES[pred_class], confidence

# ============================================================
# DEMONSTRAÇÃO COM AMOSTRAS DO DATASET
# ============================================================
print("📌 Demonstração do Protótipo de Classificação\n")

demo_images = [
    os.path.join(BASE_PATH, 'normal_ecg_images', 'Normal(5).jpg'),
    os.path.join(BASE_PATH, 'myocardial_infarction_ecg_images', 'MI(5).jpg'),
    os.path.join(BASE_PATH, 'abnormal_heartbeat_ecg_images', 'HB(5).jpg'),
    os.path.join(BASE_PATH, 'post_mi_history_ecg_images', 'PMI(5).jpg'),
]

for img_path in demo_images:
    if os.path.exists(img_path):
        classificar_ecg(img_path)
        print("\n" + "─" * 60 + "\n")
    else:
        print(f"⚠️ Imagem não encontrada: {img_path}")

---
## 10. 📝 Conclusão <a id="conclusion"></a>

### Resultados Alcançados

Neste notebook, desenvolvemos um protótipo completo de **Assistente Cardiológico Virtual** com Visão Computacional, cumprindo os objetivos propostos:

#### Parte 1 — Pré-processamento
- ✅ Dataset público de imagens de ECG selecionado e carregado
- ✅ Pipeline de pré-processamento implementado (redimensionamento, normalização, conversão RGB)
- ✅ Divisão estratificada em conjuntos de treino (70%), validação (15%) e teste (15%)
- ✅ Data augmentation configurada para mitigar o tamanho limitado do dataset

#### Parte 2 — Classificação com CNN
- ✅ CNN simples treinada do zero com 3 blocos convolucionais
- ✅ Transfer Learning com VGG16 pré-treinado (ImageNet) + fine-tuning
- ✅ Avaliação completa com acurácia, precisão, recall, F1-score e matrizes de confusão
- ✅ Protótipo interativo de classificação com visualização dos resultados

### Observações Técnicas

- O dataset utilizado possui apenas **120 imagens** (30 por classe), o que limita a capacidade de generalização. Para aplicações reais, recomenda-se utilizar o dataset completo do Kaggle.
- O **Transfer Learning** tende a apresentar resultados superiores em cenários com poucos dados, pois aproveita features aprendidas em milhões de imagens.
- O **Data Augmentation** foi essencial para mitigar overfitting neste cenário de dados limitados.

### Aplicação Clínica

Este protótipo demonstra como técnicas de Visão Computacional podem auxiliar na **triagem automatizada de ECGs**, servindo como ferramenta de suporte à decisão clínica. Em um cenário real, o sistema poderia:
- Priorizar exames com padrões anormais para revisão médica urgente
- Reduzir o tempo de análise em ambientes com alto volume de exames
- Complementar a avaliação do cardiologista